# Aria Disaster Recovery & Incident Management Guide

**Complete guide for disaster recovery planning, incident response, runbooks, and post-mortems.**

Learn how to prevent, respond to, and recover from system failures.


## Disaster Recovery Strategy

### RTO & RPO Targets

```
Recovery Objectives
├─ RTO (Recovery Time Objective): 15 minutes
│  └─ Maximum acceptable downtime after failure
│
├─ RPO (Recovery Point Objective): 1 hour
│  └─ Maximum acceptable data loss
│
└─ MTTR (Mean Time To Recovery): 10 minutes
   └─ Average time to recover from incidents
```

### DR Infrastructure

```
Primary Region (US-East)
├─ Production Database
├─ Application Instances
├─ Redis Cache
└─ File Storage
    ↓ (async replication, 1-hour lag)
Secondary Region (US-West)
├─ Standby Database
├─ Standby App Instances
├─ Cache Replica
└─ Storage Replica
```

### Business Continuity Plan

```python
# scripts/dr_manager.py
import asyncio
from datetime import datetime

class DisasterRecoveryManager:
    def __init__(self, primary_db: str, secondary_db: str):
        self.primary = create_engine(primary_db)
        self.secondary = create_engine(secondary_db)
        self.is_failing_over = False

    async def detect_failure(self) -> bool:
        """Detect if primary is unavailable."""
        try:
            with self.primary.connect() as conn:
                conn.execute("SELECT 1")
            return False  # Primary is healthy
        except Exception as e:
            print(f"❌ Primary failure detected: {e}")
            return True

    async def initiate_failover(self):
        """Failover to secondary."""
        if self.is_failing_over:
            return

        self.is_failing_over = True
        print("🚨 FAILOVER INITIATED")

        try:
            # Step 1: Verify secondary is ready
            with self.secondary.connect() as conn:
                result = conn.execute("SELECT version()")
                print(f"✓ Secondary DB online: {result.scalar()}")

            # Step 2: Point application to secondary
            os.environ["QAI_DB_CONN"] = os.getenv("SECONDARY_DB_URL")
            print("✓ App switched to secondary database")

            # Step 3: Notify operations team
            await self.notify_team("Failover completed - running on secondary")

            # Step 4: Begin recovery of primary
            await self.recover_primary()

        except Exception as e:
            print(f"❌ Failover failed: {e}")
            await self.notify_team(f"FAILOVER FAILED: {e}", severity="critical")
            raise

    async def recover_primary(self):
        """Attempt to recover and re-sync primary."""
        max_attempts = 3
        for attempt in range(max_attempts):
            try:
                print(f"[{attempt+1}/{max_attempts}] Attempting to recover primary...")

                # Try to connect
                with self.primary.connect() as conn:
                    conn.execute("SELECT 1")

                # Re-sync data
                await self.resync_primary()
                print("✓ Primary recovered and re-synced")
                return

            except Exception as e:
                print(f"❌ Recovery attempt {attempt+1} failed: {e}")
                await asyncio.sleep(60)  # Wait before retry

        print("❌ Primary recovery failed - will remain on secondary")

    async def resync_primary(self):
        """Resync primary with secondary data."""
        # Query secondary for changes since failure
        # Replay changes to primary
        pass

# Monitor for failures
async def run_health_check():
    dr = DisasterRecoveryManager(
        os.getenv("PRIMARY_DB_URL"),
        os.getenv("SECONDARY_DB_URL")
    )

    while True:
        if await dr.detect_failure():
            await dr.initiate_failover()

        await asyncio.sleep(60)  # Check every minute
```

---

## Incident Response

### Incident Classification

```
Severity Levels
├─ SEV-1 (Critical)
│  └─ Complete service outage, no workaround
│     └─ Action: Page on-call immediately
│
├─ SEV-2 (High)
│  └─ Significant functionality impaired
│     └─ Action: Alert team within 5 minutes
│
├─ SEV-3 (Medium)
│  └─ Minor degradation or isolated failures
│     └─ Action: Log and monitor
│
└─ SEV-4 (Low)
   └─ Small issues with workarounds
      └─ Action: Backlog for next sprint
```

### Incident Response Runbook

#### **Runbook: Database Connection Pool Exhausted**

```
SYMPTOM: /api/chat returns 503, SQL pool at 100%

IMMEDIATE (0-5 min)
1. Check dashboard: Go to Azure Monitor → Database Pool Metrics
2. Verify: Is primary DB responsive?
   - Run: SELECT COUNT(*) FROM pg_stat_activity;
   - If >100 connections: STEP 3
   - If DB unresponsive: Go to Failover Runbook

DIAGNOSIS (5-15 min)
3. Identify long-running queries:
   SELECT pid, usename, state, query_start, query
   FROM pg_stat_activity
   WHERE state = 'active'
   AND (NOW() - query_start) > INTERVAL '5 minutes'
   ORDER BY query_start DESC;

4. Check connection leak:
   SELECT usename, count(*)
   FROM pg_stat_activity
   GROUP BY usename;

REMEDIATION (15-30 min)
5. If long query found:
   - Analyze: EXPLAIN ANALYZE SELECT ...
   - Kill if safe: SELECT pg_terminate_backend(pid);

6. If connection leak:
   - Check application logs for errors
   - Restart affected service instances
   - Command: az container restart --name aria-api --resource-group aria-prod

7. Verify recovery:
   - Monitor connection count for 5 minutes
   - Ensure /api/ai/status returns OK

ESCALATION
- If not resolved in 30 min: Call DBA on-call
- If data corruption: Begin failover
```

#### **Runbook: API Latency Spike**

```
SYMPTOM: /api/chat_latency_p95 > 5000ms

IMMEDIATE (0-5 min)
1. Check related metrics:
   - CPU usage: Should be <70%
   - Memory: Should be <80%
   - DB connections: Should be <80%
   - Cache hit rate: Should be >80%

DIAGNOSIS (5-15 min)
2. If CPU high (>70%):
   a. Check if training job is running
      ps aux | grep train
   b. Check hot functions:
      Performance profiler in App Insights

3. If Memory high (>80%):
   a. Check for memory leak
      systemctl status aria-api
   b. Compare heap size vs normal

4. If DB connections high:
   See: Database Connection Pool Exhausted

5. If cache hit rate low (<50%):
   Redis restart needed

REMEDIATION (15-30 min)
6. Scale up instances:
   az appservice plan update --name aria-plan --sku P2V2

7. Clear cache if needed:
   redis-cli FLUSHALL

8. Monitor improvement for 10 minutes

ESCALATION
- If not improving: Roll back recent code changes
- Contact: Platform on-call engineer
```

#### **Runbook: Model Inference Errors**

```
SYMPTOM: /api/quantum/run returns 500, or /api/chat uses wrong provider

IMMEDIATE (0-5 min)
1. Check provider status:
   curl http://localhost:7071/api/ai/status

2. Verify which provider is active:
   - Azure OpenAI?
   - OpenAI API?
   - LMStudio local?
   - LoRA adapter?

DIAGNOSIS (5-15 min)
3. Check provider-specific issues:

   Azure OpenAI:
   - Verify AZURE_OPENAI_API_KEY is set
   - Check rate limits: az cognitiveservices account list

   OpenAI:
   - Verify OPENAI_API_KEY is set
   - Check account balance

   LMStudio:
   - Check if service running: ps aux | grep LM
   - Test: curl http://localhost:1234/v1/models

   LoRA:
   - Verify adapter files exist
   - Check: ls -la checkpoints/adapter/

4. Check recent provider changes:
   git log --oneline -10 shared/chat_providers.py

REMEDIATION (15-30 min)
5. If provider unreachable:
   - Fallback to local provider
   - Set: DEFAULT_AI_PROVIDER=local
   - Restart: systemctl restart aria-api

6. If model file corrupted:
   - Restore from backup: aws s3 cp s3://aria-backups/model.bin .
   - Restart service

7. Test: curl -X POST http://localhost:7071/api/chat -d '{"message":"test"}'

ESCALATION
- If no provider available: Page ML on-call
- Notify users of degraded service
```

---

## Post-Incident Analysis

### Post-Mortem Template

````markdown
# Post-Mortem: [Incident Title]

## Incident Summary

- **Date/Time**: 2024-01-15 14:30 UTC
- **Duration**: 45 minutes
- **Severity**: SEV-2
- **Status**: Resolved

## Impact

- **Users Affected**: ~5,000
- **Revenue Lost**: ~$500
- **Errors Generated**: ~2,000
- **Percentage Downtime**: 0.03%

## Timeline

- 14:30 - Monitoring alert triggered (high DB connections)
- 14:32 - On-call engineer paged
- 14:35 - Root cause identified (connection leak in new code)
- 14:40 - Rollback initiated
- 14:45 - Service recovered
- 14:50 - All-clear signal

## Root Cause Analysis

The recent deployment (PR #1234) introduced a connection leak in the chat provider.
Each message would open a connection but not properly close it on error cases.

```python
# Before (BAD)
async def chat(prompt):
    conn = await get_db_connection()
    # ... but no finally block to close on error

# After (GOOD)
async def chat(prompt):
    conn = await get_db_connection()
    try:
        # ... logic ...
    finally:
        await conn.close()
```
````

## Contributing Factors

1. Connection pooling tests were missing
2. Integration tests didn't simulate connection failures
3. Code review didn't catch resource cleanup

## Lessons Learned

1. **What went well**: Automated monitoring detected the issue immediately
2. **What went poorly**: Connection pool tests not run in CI
3. **What surprised us**: Rollback took longer than expected due to cache invalidation

## Action Items

| Item                               | Owner          | Due Date   |
| ---------------------------------- | -------------- | ---------- |
| Add connection pooling tests to CI | @alice         | 2024-01-17 |
| Review all resource cleanup code   | @bob           | 2024-01-17 |
| Implement connection pool metrics  | @charlie       | 2024-01-20 |
| Improve rollback automation        | @platform-team | 2024-01-24 |

## Prevention

- Mandatory integration tests for DB connection handling
- Code review checklist item for resource cleanup
- New monitoring alerts for connection pool near-saturation

```

### Blameless Culture

```

Instead of "Why did Alice deploy broken code?"
Ask: "Why did our systems allow broken code to reach production?"

Key Principles:

1. Assume good intent
2. Focus on systems, not people
3. Learn, don't punish
4. Publish post-mortems
5. Implement improvements
6. Follow up on action items

```

```


## Backup & Restore Procedures

### Automated Backup Schedule

```yaml
# config/backup_config.yaml
backup_policy:
    databases:
        primary:
            schedule: "0 2 * * *" # Daily at 2 AM
            retention: 30 # Keep 30 days
            destination: "s3://aria-backups/databases/"

    files:
        schedule: "0 3 * * *" # Daily at 3 AM
        retention: 7
        destination: "s3://aria-backups/files/"

    models:
        schedule: "0 4 * * *" # Daily at 4 AM
        retention: 30
        destination: "s3://aria-backups/models/"

verification:
    enabled: true
    schedule: "0 6 * * *" # Verify at 6 AM
    test_restore: true
    notify_on_failure: true
```

### Restore Procedure

```bash
#!/bin/bash
# scripts/restore_from_backup.sh

BACKUP_FILE=$1
TARGET_DB=$2

if [ -z "$BACKUP_FILE" ] || [ -z "$TARGET_DB" ]; then
    echo "Usage: ./restore_from_backup.sh <backup_file> <target_db>"
    exit 1
fi

echo "🔄 Starting restore from $BACKUP_FILE..."

# Step 1: Verify backup integrity
echo "1/4: Verifying backup integrity..."
pg_restore --list $BACKUP_FILE > /tmp/backup_contents.txt
if [ $? -ne 0 ]; then
    echo "❌ Backup verification failed"
    exit 1
fi

# Step 2: Create temporary database for testing
echo "2/4: Testing restore..."
createdb test_restore_$(date +%s)
pg_restore -d test_restore_* $BACKUP_FILE
if [ $? -ne 0 ]; then
    echo "❌ Test restore failed"
    exit 1
fi
dropdb test_restore_*

# Step 3: Perform actual restore
echo "3/4: Restoring to $TARGET_DB..."
pg_restore -d $TARGET_DB $BACKUP_FILE
if [ $? -ne 0 ]; then
    echo "❌ Restore failed"
    exit 1
fi

# Step 4: Verify restored data
echo "4/4: Verifying restored data..."
RECORD_COUNT=$(psql -d $TARGET_DB -t -c "SELECT COUNT(*) FROM chat_messages;")
echo "✓ Restored database contains $RECORD_COUNT chat messages"

echo "✅ Restore completed successfully"
```

---

## Incident Management Checklist

### Response

- [ ] Acknowledge alert within 5 minutes
- [ ] Determine severity level
- [ ] Page appropriate on-call person
- [ ] Create incident ticket
- [ ] Post updates every 15 minutes
- [ ] Keep stakeholders informed

### During Incident

- [ ] Follow runbook procedures
- [ ] Collect logs and metrics
- [ ] Document all actions taken
- [ ] Communicate ETA
- [ ] Prepare rollback plan
- [ ] Monitor for side effects

### Resolution

- [ ] Verify all systems healthy
- [ ] Run smoke tests
- [ ] Confirm customer impact resolved
- [ ] Close incident ticket
- [ ] Schedule post-mortem
- [ ] Send all-clear notification

### Post-Incident

- [ ] Complete root cause analysis
- [ ] Write post-mortem report
- [ ] Assign action items
- [ ] Implement immediate fixes
- [ ] Plan systemic improvements
- [ ] Schedule follow-up (1 month)
